# Аналіз якості Wi-Fi за допомогою власної багатошарової перцептронної мережі (MLP)

Нейронна мережа визначає результат **0 або 1** (погана / хороша якість Wi-Fi)
на основі трьох параметрів:

- **Download** – швидкість завантаження
- **Upload** – швидкість відвантаження
- **PacketLoss** – втрата пакетів

Мережа реалізована власноруч на **pandas** (без sklearn/pytorch):
прямий прохід, зворотне поширення помилки та оновлення ваг.

## 1. Імпорти та завантаження даних

Тренувальні дані беремо з файлу `wifi_training.csv`.

In [1]:
import math
import random

import pandas as pd

df = pd.read_csv("wifi_training.csv")
print(f"Рядків у наборі даних: {len(df)}")
df.head(10)

Рядків у наборі даних: 48


,Download,Upload,PacketLoss,Expected
0,5,1,8.5,0
1,8,2,6.2,0
2,10,2,5.8,0
3,12,3,4.7,0
4,15,3,4.1,0
5,18,4,3.8,0
6,20,5,3.2,0
7,22,4,2.9,0
8,25,5,2.5,0
9,28,6,2.1,0


## 2. Підготовка даних

Формуємо вхідні ознаки `X` та цільову змінну `y`
(стовпець `Expected` перейменовуємо на `O1` — ім'я вихідного нейрона).

In [2]:
X = df[["Download", "Upload", "PacketLoss"]]
y = df[["Expected"]].rename(columns={"Expected": "O1"})

print("X — вхідні ознаки:")
display(X.head())
print("y — цільова змінна:")
display(y.head())

X — вхідні ознаки:


,Download,Upload,PacketLoss
0,5,1,8.5
1,8,2,6.2
2,10,2,5.8
3,12,3,4.7
4,15,3,4.1


y — цільова змінна:


,O1
0,0
1,0
2,0
3,0
4,0


## 3. Min-Max нормалізація вхідних даних

Приводимо кожну ознаку до діапазону **[0, 1]** за формулою:

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Мінімуми та максимуми зберігаємо — вони знадобляться для нормалізації
нових даних (`wifi_analysis.csv`) за **тими самими** параметрами.

In [3]:
X_min = X.min()
X_max = X.max()

X_norm = (X - X_min) / (X_max - X_min)
X_norm.head()

,Download,Upload,PacketLoss
0,0.000000,0.000000,1.000000
1,0.020690,0.038462,0.729412
2,0.034483,0.038462,0.682353
3,0.048276,0.076923,0.552941
4,0.068966,0.076923,0.482353


## 4. Власна багатошарова перцептронна мережа

Архітектура: **3 входи → 4 приховані нейрони → 1 вихід**.

- Ваги `W1`, `W2` та зміщення `b1`, `b2` зберігаються як таблиці pandas.
- Функція активації — **сигмоїда** (вихід у діапазоні 0..1, схожий на ймовірність).
- Навчання — градієнтний спуск зі зворотним поширенням помилки.

In [4]:
class NeuralNetwork:
    """
    Нейронна мережа не містить окремих нейронів — лише окремі шари нейронів.
    Є ваги і bias між шарами.

    Ваги між шарами визначають, наскільки сильно кожен нейрон попереднього
    шару впливає на кожен нейрон наступного шару.
    На кожну вагу використовується окремий показник ваги.

    Між шарами знаходяться ваги W1 і W2, які є таблицями pandas.

    | Вага | Вплив                    |
    | -----|--------------------------|
    | +0.9 | дуже сильний позитивний  |
    | +0.2 | слабкий позитивний       |
    |  0   | практично немає впливу   |
    | -0.2 | слабкий негативний       |
    | -0.9 | дуже сильний негативний  |
    """

    def __init__(
        self,
        input_size=3,       # Розмір вхідного шару нейронів
        hidden_size=4,      # Розмір прихованого шару нейронів
        output_size=1,      # Розмір вихідного шару нейронів
        learning_rate=0.1
    ):
        random.seed(42)
        self.learning_rate = learning_rate

        # input_size=3   Вхідний шар має 3 нейрони
        # hidden_size=4  Прихований шар — 4 нейрони по 3 входи
        # output_size=1  Вихідний шар — 1 нейрон на 4 входи
        self.W1 = pd.DataFrame(
            # random.gauss() — встановлення випадкових початкових ваг
            [
                [random.gauss(0, 0.1) for _ in range(hidden_size)]
                for _ in range(input_size)
            ],
            index=["Download", "Upload", "PacketLoss"],
            columns=[f"H{i + 1}" for i in range(hidden_size)]
        )

        # Bias прихованого шару
        self.b1 = pd.Series(
            [0.0] * hidden_size,
            index=self.W1.columns
        )

        # Ваги між прихованим і вихідним шарами
        self.W2 = pd.DataFrame(
            [
                [random.gauss(0, 0.1) for _ in range(output_size)]
                for _ in range(hidden_size)
            ],
            index=[f"H{i + 1}" for i in range(hidden_size)],
            columns=[f"O{i + 1}" for i in range(output_size)]
        )

        # Bias вихідного шару
        self.b2 = pd.Series(
            [0.0] * output_size,
            index=self.W2.columns
        )

    # Функція активації. Дає значення, схоже на ймовірність від 0 до 1.
    # Використовується в мережах, що дають бінарний вихід.
    def sigmoid(self, z):
        return z.map(
            lambda value: 1 / (1 + math.exp(-value))
        )

    # Показує, наскільки сильно потрібно змінити ваги нейрона під час навчання
    def sigmoid_derivative(self, value):
        return value * (1 - value)

    # Виконує прямий прохід даних через нейронну мережу:
    # бере вхідні дані, пропускає їх через усі шари та отримує результат.
    def forward(self, X):
        # Вхідний шар → прихований шар
        self.z1 = X.dot(self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)

        # Прихований шар → вихідний шар
        self.z2 = self.a1.dot(self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)

        return self.a2

    # Тренування нейронної мережі
    def train(self, X, y, epochs=10000):
        for epoch in range(epochs):
            # Пряме поширення
            output = self.forward(X)

            # Помилка
            error = y - output

            # Градієнт вихідного шару
            dz2 = error * self.sigmoid_derivative(output)

            # Градієнт ваг W2
            dW2 = self.a1.T.dot(dz2)

            # Градієнт bias вихідного шару
            db2 = dz2.sum(axis=0)

            # Помилка прихованого шару
            dz1 = pd.DataFrame(
                dz2.to_numpy() @ self.W2.to_numpy().T,
                index=dz2.index,
                columns=self.W2.index
            ) * self.sigmoid_derivative(self.a1)

            # Градієнт ваг W1
            dW1 = X.T.dot(dz1)

            # Градієнт bias прихованого шару
            db1 = dz1.sum(axis=0)

            # Оновлення ваг
            self.W2 += self.learning_rate * dW2
            self.b2 += self.learning_rate * db2
            self.W1 += self.learning_rate * dW1
            self.b1 += self.learning_rate * db1

            # Виводимо помилку кожні 500 епох
            if (epoch + 1) % 500 == 0:
                loss = (error ** 2).mean().mean()
                print(
                    f"Епоха {epoch + 1:5}: "
                    f"помилка = {loss:.6f}"
                )

    # Повертає ймовірнісний результат роботи нейронної мережі
    def predict_proba(self, X):
        return self.forward(X)

    # Отримує остаточний прогноз нейронної мережі: 0 або 1.
    # Для цього використовує ймовірнісний результат,
    # отриманий за допомогою функції predict_proba().
    def predict(self, X):
        probability = self.predict_proba(X)
        return probability.map(
            lambda value: 1 if value >= 0.5 else 0
        )

## 5. Створення об'єкта нейронної мережі та її навчання

Навчаємо мережу на нормалізованих даних протягом **10000 епох**.

In [5]:
nn = NeuralNetwork(
    input_size=3,
    hidden_size=4,
    output_size=1,
    learning_rate=0.1
)

nn.train(X_norm, y, epochs=10000)

Епоха   500: помилка = 0.032195


Епоха  1000: помилка = 0.017846


Епоха  1500: помилка = 0.013503


Епоха  2000: помилка = 0.011207


Епоха  2500: помилка = 0.009664


Епоха  3000: помилка = 0.008516


... (10 outputs left out)

Епоха  8500: помилка = 0.003379


Епоха  9000: помилка = 0.003166


Епоха  9500: помилка = 0.002972


Епоха 10000: помилка = 0.002795


## 6. Перевірка на тренувальних даних

Як читати стовпець **Probability**:

| Probability | Інтерпретація                        |
|-------------|--------------------------------------|
| 0.02        | нейрон майже впевнений, що це 0      |
| 0.08        | нейрон схиляється до 0               |
| 0.71        | нейрон схиляється до 1               |
| 0.96        | нейрон майже впевнений, що це 1      |

In [6]:
results = X.copy()
results["Expected"] = df["Expected"]
results["Probability"] = nn.predict_proba(X_norm)["O1"].round(3)
results["Prediction"] = nn.predict(X_norm)["O1"]
results["Correct"] = results["Prediction"] == results["Expected"]

print(f"Точність на тренувальних даних: {results['Correct'].mean():.1%}")
results

Точність на тренувальних даних: 100.0%


,Download,Upload,PacketLoss,Expected,Probability,Prediction,Correct
0,5,1,8.5,0,0.000,0,True
1,8,2,6.2,0,0.000,0,True
2,10,2,5.8,0,0.000,0,True
3,12,3,4.7,0,0.000,0,True
4,15,3,4.1,0,0.000,0,True
5,18,4,3.8,0,0.000,0,True
6,20,5,3.2,0,0.000,0,True
7,22,4,2.9,0,0.000,0,True
8,25,5,2.5,0,0.000,0,True
9,28,6,2.1,0,0.000,0,True


## 7. Перевірка на нових даних (`wifi_analysis.csv`)

Важливо: нові дані нормалізуємо за **тими самими** `X_min` / `X_max`,
що й тренувальні (мережа "бачить" дані в тому ж масштабі, на якому вчилася).

Файл `wifi_analysis.csv` не містить стовпця `Expected`, тому еталонну мітку
відновлюємо за правилом, яке на 100% описує тренувальні дані:
**хороший Wi-Fi ⇔ Download ≥ 45 та Upload ≥ 6**.
Воно потрібне лише для обчислення стовпця `Correct`.

In [7]:
df_new = pd.read_csv("wifi_analysis.csv")
X_new = df_new[["Download", "Upload", "PacketLoss"]]

# Нормалізація за параметрами тренувальної вибірки
X_new_norm = (X_new - X_min) / (X_max - X_min)

# Еталонні мітки за правилом, виведеним з тренувальних даних
expected_new = ((X_new["Download"] >= 45) & (X_new["Upload"] >= 6)).astype(int)

results_new = X_new.copy()
results_new["Expected"] = expected_new
results_new["Probability"] = nn.predict_proba(X_new_norm)["O1"].round(3)
results_new["Prediction"] = nn.predict(X_new_norm)["O1"]
results_new["Correct"] = results_new["Prediction"] == results_new["Expected"]

print(f"Збіг з еталонним правилом: {results_new['Correct'].mean():.1%}")
results_new

Збіг з еталонним правилом: 100.0%


,Download,Upload,PacketLoss,Expected,Probability,Prediction,Correct
0,7,1,7.8,0,0.000,0,True
1,11,2,5.5,0,0.000,0,True
2,17,3,4.5,0,0.000,0,True
3,21,4,3.6,0,0.000,0,True
4,27,5,2.8,0,0.000,0,True
5,33,6,2.0,0,0.000,0,True
6,39,7,1.4,0,0.027,0,True
7,42,6,2.5,0,0.075,0,True
8,47,7,1.9,1,0.925,1,True
9,52,8,1.6,1,0.999,1,True


## Висновки

- Власна MLP-мережа (3 → 4 → 1) на pandas успішно навчилася розрізняти
  хороший та поганий Wi-Fi: середньоквадратична помилка за 10000 епох
  зменшилася до тисячних часток.
- На тренувальних даних мережа дає **100% правильних прогнозів** —
  межа прийняття рішення відповідає правилу `Download ≥ 45 і Upload ≥ 6`.
- На нових даних з `wifi_analysis.csv` прогнози узгоджуються з еталонним
  правилом, отже мережа **узагальнює** закономірність, а не запам'ятовує
  тренувальну вибірку.